In [1]:
# 4.3 Factor Zoo: import full trading + partial fundamentals, then build factors into new DuckDB
from pathlib import Path
import duckdb
import pandas as pd

print("=" * 72)
print("4.3 BUILD FACTOR ZOO INTO NEW DUCKDB")
print("=" * 72)

# ---- Paths ----
src_candidates = [Path("../trading_data.duckdb"), Path("trading_data.duckdb"), Path.cwd() / "trading_data.duckdb"]
src_db = next((p.resolve() for p in src_candidates if p.exists()), src_candidates[0].resolve())
dst_db = Path("factor_zoo.duckdb").resolve()

print(f"Source DuckDB: {src_db}")
print(f"Target DuckDB: {dst_db}")

# ---- Inspect source schema ----
src = duckdb.connect(str(src_db), read_only=True)
tables_df = src.execute("SHOW TABLES").fetchdf()
if tables_df.empty:
    raise ValueError("Source DuckDB has no tables.")
tables = tables_df.iloc[:, 0].astype(str).tolist()

def table_columns(con, table_name: str):
    info = con.execute(f"PRAGMA table_info('{table_name}')").fetchdf()
    return info["name"].astype(str).tolist()

table_cols = {t: table_columns(src, t) for t in tables}

preferred_tables = ["trading_data_clean", "trading_data_en", "trading_data_raw"]
trade_table = None
for t in preferred_tables:
    if t in table_cols and {"trade_date", "stock_code"}.issubset(set(table_cols[t])):
        trade_table = t
        break
if trade_table is None:
    for t, cols in table_cols.items():
        if {"trade_date", "stock_code"}.issubset(set(cols)):
            trade_table = t
            break
if trade_table is None:
    raise ValueError("No trading-like table found (requires trade_date and stock_code).")

trade_cols = set(table_cols[trade_table])
print(f"Main trading table: {trade_table}")
print(f"Columns in main table: {len(trade_cols)}")

fund_keywords = [
    "pe", "pb", "ps", "pcf", "div", "yield", "mv", "cap", "roe", "roa",
    "margin", "profit", "revenue", "eps", "bps", "debt", "asset", "liability",
    "cash", "book", "turnover", "valuation"
 ]
fund_tables = []
for t, cols in table_cols.items():
    low = " ".join([t.lower()] + [c.lower() for c in cols])
    if any(k in low for k in fund_keywords) and t != trade_table:
        fund_tables.append(t)

dst = duckdb.connect(str(dst_db))
dst.execute("PRAGMA threads=4")
try:
    dst.execute("DETACH srcdb")
except Exception:
    pass
dst.execute("ATTACH '" + str(src_db).replace('\\', '/') + "' AS srcdb (READ_ONLY)")

dst.execute(f"CREATE OR REPLACE TABLE trading_data_all AS SELECT * FROM srcdb.{trade_table}")
for t in fund_tables:
    dst.execute(f"CREATE OR REPLACE TABLE raw_{t} AS SELECT * FROM srcdb.{t}")

def pick_expr(candidates, alias, cast_type="DOUBLE") -> str:
    for c in candidates:
        if c in trade_cols:
            return f"CAST({c} AS {cast_type}) AS {alias}"
    return f"CAST(NULL AS {cast_type}) AS {alias}"

stock_name_expr = "stock_name" if "stock_name" in trade_cols else "CAST(NULL AS VARCHAR) AS stock_name"
open_expr = pick_expr(["open", "open_price"], "open", "DOUBLE")
high_expr = pick_expr(["high", "high_price"], "high", "DOUBLE")
low_expr = pick_expr(["low", "low_price"], "low", "DOUBLE")
close_expr = pick_expr(["close", "close_price"], "close", "DOUBLE")
prev_close_expr = pick_expr(["prev_close", "pre_close"], "prev_close", "DOUBLE")
chg_expr = pick_expr(["change_pct", "pct_chg", "return_pct"], "change_pct", "DOUBLE")
vol_expr = pick_expr(["volume", "vol"], "volume", "DOUBLE")
amt_expr = pick_expr(["amount", "turnover_amount"], "amount", "DOUBLE")
turn_expr = pick_expr(["turnover_rate", "turnover"], "turnover_rate", "DOUBLE")
pe_expr = pick_expr(["pe_ttm", "pe"], "pe_ttm", "DOUBLE")
pb_expr = pick_expr(["pb", "pb_lf"], "pb", "DOUBLE")
ps_expr = pick_expr(["ps_ttm", "ps"], "ps_ttm", "DOUBLE")
pcf_expr = pick_expr(["pcf_ttm", "pcf"], "pcf_ttm", "DOUBLE")
dy_expr = pick_expr(["dividend_yield", "div_yield", "dv_ttm"], "dividend_yield", "DOUBLE")
mv_expr = pick_expr(["total_mv", "market_cap", "total_market_cap"], "total_mv", "DOUBLE")
float_mv_expr = pick_expr(["circ_mv", "float_mv", "free_float_mv"], "float_mv", "DOUBLE")

normalize_sql = f"""
CREATE OR REPLACE TABLE factor_base AS
SELECT
    CAST(trade_date AS DATE) AS trade_date,
    LPAD(REGEXP_EXTRACT(COALESCE(stock_code, ''), '([0-9]+)', 1), 6, '0') AS stock_code,
    {stock_name_expr},
    {open_expr},
    {high_expr},
    {low_expr},
    {close_expr},
    {prev_close_expr},
    {chg_expr},
    {vol_expr},
    {amt_expr},
    {turn_expr},
    {pe_expr},
    {pb_expr},
    {ps_expr},
    {pcf_expr},
    {dy_expr},
    {mv_expr},
    {float_mv_expr}
FROM trading_data_all
WHERE trade_date IS NOT NULL
"""
dst.execute(normalize_sql)

dst.execute("""
CREATE OR REPLACE TABLE fundamentals_snapshot AS
SELECT
    trade_date, stock_code, stock_name,
    pe_ttm, pb, ps_ttm, pcf_ttm, dividend_yield, total_mv, float_mv, turnover_rate
FROM factor_base
""")

factor_sql = """
CREATE OR REPLACE TABLE factor_zoo_daily AS
WITH base AS (
    SELECT
        trade_date, stock_code, stock_name,
        open, high, low, close, prev_close,
        volume, amount, turnover_rate,
        pe_ttm, pb, ps_ttm, pcf_ttm, dividend_yield, total_mv, float_mv,
        COALESCE(
            change_pct / 100.0,
            CASE WHEN prev_close IS NOT NULL AND prev_close <> 0 AND close IS NOT NULL
                 THEN (close - prev_close) / prev_close
                 ELSE NULL
            END
        ) AS ret_1d
    FROM factor_base
    WHERE stock_code IS NOT NULL AND stock_code <> ''
),
mkt AS (
    SELECT trade_date, ret_1d AS hs300_ret
    FROM base WHERE stock_code = '000300'
),
joined AS (
    SELECT b.*, m.hs300_ret
    FROM base b
    LEFT JOIN mkt m USING(trade_date)
),
roll AS (
    SELECT
        *,
        LAG(close, 5) OVER w AS close_l5,
        LAG(close, 10) OVER w AS close_l10,
        LAG(close, 20) OVER w AS close_l20,
        LAG(close, 60) OVER w AS close_l60,
        LAG(close, 120) OVER w AS close_l120,
        LAG(close, 252) OVER w AS close_l252,
        AVG(close) OVER w5 AS ma_5,
        AVG(close) OVER w10 AS ma_10,
        AVG(close) OVER w20 AS ma_20,
        AVG(close) OVER w60 AS ma_60,
        AVG(close) OVER w120 AS ma_120,
        STDDEV_SAMP(ret_1d) OVER w20 AS vol_20d,
        STDDEV_SAMP(ret_1d) OVER w60 AS vol_60d,
        STDDEV_SAMP(ret_1d) OVER w120 AS vol_120d,
        AVG(CASE WHEN ret_1d < 0 THEN ret_1d * ret_1d ELSE NULL END) OVER w20 AS downside_var_20d,
        AVG(turnover_rate) OVER w20 AS turnover_mean_20d,
        STDDEV_SAMP(turnover_rate) OVER w20 AS turnover_std_20d,
        AVG(volume) OVER w20 AS volume_mean_20d,
        AVG(ABS(ret_1d) / NULLIF(amount, 0)) OVER w20 AS amihud_20d,
        COVAR_SAMP(ret_1d, hs300_ret) OVER w60 AS cov_mkt_60d,
        VAR_SAMP(hs300_ret) OVER w60 AS var_mkt_60d
    FROM joined
    WINDOW
        w AS (PARTITION BY stock_code ORDER BY trade_date),
        w5 AS (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        w10 AS (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW),
        w20 AS (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
        w60 AS (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
        w120 AS (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW)
),
fac AS (
    SELECT
        trade_date, stock_code, stock_name, ret_1d, close, hs300_ret,
        (close / NULLIF(close_l5, 0) - 1) AS mom_5d,
        (close / NULLIF(close_l10, 0) - 1) AS mom_10d,
        (close / NULLIF(close_l20, 0) - 1) AS mom_20d,
        (close / NULLIF(close_l60, 0) - 1) AS mom_60d,
        (close / NULLIF(close_l120, 0) - 1) AS mom_120d,
        (close / NULLIF(close_l252, 0) - 1) AS mom_252d,
        -1.0 * (close / NULLIF(close_l5, 0) - 1) AS rev_5d,
        -1.0 * (close / NULLIF(close_l20, 0) - 1) AS rev_20d,
        (close / NULLIF(ma_5, 0) - 1) AS close_ma5_gap,
        (close / NULLIF(ma_10, 0) - 1) AS close_ma10_gap,
        (close / NULLIF(ma_20, 0) - 1) AS close_ma20_gap,
        (close / NULLIF(ma_60, 0) - 1) AS close_ma60_gap,
        (ma_5 / NULLIF(ma_20, 0) - 1) AS ma5_ma20_spread,
        (ma_20 / NULLIF(ma_60, 0) - 1) AS ma20_ma60_spread,
        (high - low) / NULLIF(close, 0) AS intraday_range,
        (close - open) / NULLIF(open, 0) AS intraday_return,
        vol_20d, vol_60d, vol_120d, SQRT(NULLIF(downside_var_20d, 0)) AS downside_vol_20d,
        CASE WHEN var_mkt_60d IS NULL OR var_mkt_60d = 0 THEN NULL ELSE cov_mkt_60d / var_mkt_60d END AS beta_60d,
        turnover_rate, turnover_mean_20d, turnover_std_20d, volume_mean_20d, amihud_20d,
        pe_ttm, pb, ps_ttm, pcf_ttm, dividend_yield, total_mv, float_mv,
        CASE WHEN pb IS NULL OR pb = 0 THEN NULL ELSE 1.0 / pb END AS book_to_price,
        CASE WHEN pe_ttm IS NULL OR pe_ttm = 0 THEN NULL ELSE 1.0 / pe_ttm END AS earnings_yield,
        CASE WHEN ps_ttm IS NULL OR ps_ttm = 0 THEN NULL ELSE 1.0 / ps_ttm END AS sales_yield,
        CASE WHEN pcf_ttm IS NULL OR pcf_ttm = 0 THEN NULL ELSE 1.0 / pcf_ttm END AS cashflow_yield,
        LN(NULLIF(total_mv, 0)) AS size_ln_mv,
        POWER(LN(NULLIF(total_mv, 0)), 2) AS size_ln_mv_sq
    FROM roll
    WHERE stock_code <> '000300'
),
fac2 AS (
    SELECT
        *,
        CASE WHEN beta_60d IS NULL OR hs300_ret IS NULL THEN NULL ELSE (ret_1d - beta_60d * hs300_ret) END AS idio_return_proxy
    FROM fac
),
fac3 AS (
    SELECT
        *,
        STDDEV_SAMP(idio_return_proxy) OVER (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS idio_vol_60d,
        AVG(ret_1d) OVER (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 251 PRECEDING AND CURRENT ROW)
        / NULLIF(STDDEV_SAMP(ret_1d) OVER (PARTITION BY stock_code ORDER BY trade_date ROWS BETWEEN 251 PRECEDING AND CURRENT ROW), 0)
        AS sharpe_252d
    FROM fac2
),
fac4 AS (
    SELECT
        *,
        (ret_1d - AVG(ret_1d) OVER (PARTITION BY trade_date)) / NULLIF(STDDEV_SAMP(ret_1d) OVER (PARTITION BY trade_date), 0) AS ret_1d_cs_z,
        (mom_20d - AVG(mom_20d) OVER (PARTITION BY trade_date)) / NULLIF(STDDEV_SAMP(mom_20d) OVER (PARTITION BY trade_date), 0) AS mom_20d_cs_z,
        (vol_20d - AVG(vol_20d) OVER (PARTITION BY trade_date)) / NULLIF(STDDEV_SAMP(vol_20d) OVER (PARTITION BY trade_date), 0) AS vol_20d_cs_z,
        (size_ln_mv - AVG(size_ln_mv) OVER (PARTITION BY trade_date)) / NULLIF(STDDEV_SAMP(size_ln_mv) OVER (PARTITION BY trade_date), 0) AS size_cs_z
    FROM fac3
),
fac5 AS (
    SELECT
        *,
        NTILE(10) OVER (PARTITION BY trade_date ORDER BY mom_20d) AS mom_20d_decile,
        NTILE(10) OVER (PARTITION BY trade_date ORDER BY book_to_price) AS value_decile
    FROM fac4
 )
SELECT
    trade_date, stock_code, stock_name,
    ret_1d,
    mom_5d, mom_10d, mom_20d, mom_60d, mom_120d, mom_252d,
    rev_5d, rev_20d,
    close_ma5_gap, close_ma10_gap, close_ma20_gap, close_ma60_gap, ma5_ma20_spread, ma20_ma60_spread,
    intraday_range, intraday_return,
    vol_20d, vol_60d, vol_120d, downside_vol_20d, beta_60d, idio_vol_60d, sharpe_252d,
    turnover_rate, turnover_mean_20d, turnover_std_20d, volume_mean_20d, amihud_20d,
    pe_ttm, pb, ps_ttm, pcf_ttm, dividend_yield, total_mv, float_mv,
    book_to_price, earnings_yield, sales_yield, cashflow_yield,
    size_ln_mv, size_ln_mv_sq,
    ret_1d_cs_z, mom_20d_cs_z, vol_20d_cs_z, size_cs_z,
    mom_20d_decile, value_decile
FROM fac5
"""
dst.execute(factor_sql)

dst.execute("""
CREATE OR REPLACE TABLE factor_metadata (
    factor_name VARCHAR,
    category VARCHAR,
    formula_hint VARCHAR
)
""")

metadata_rows = [
    ("ret_1d", "return", "daily return from change_pct or (close-prev_close)/prev_close"),
    ("mom_5d", "momentum", "close/lag(close,5)-1"),
    ("mom_10d", "momentum", "close/lag(close,10)-1"),
    ("mom_20d", "momentum", "close/lag(close,20)-1"),
    ("mom_60d", "momentum", "close/lag(close,60)-1"),
    ("mom_120d", "momentum", "close/lag(close,120)-1"),
    ("mom_252d", "momentum", "close/lag(close,252)-1"),
    ("rev_5d", "reversal", "-mom_5d"),
    ("rev_20d", "reversal", "-mom_20d"),
    ("close_ma20_gap", "trend", "close/ma20-1"),
    ("ma5_ma20_spread", "trend", "ma5/ma20-1"),
    ("intraday_range", "microstructure", "(high-low)/close"),
    ("intraday_return", "microstructure", "(close-open)/open"),
    ("vol_20d", "risk", "rolling std 20d of ret_1d"),
    ("vol_60d", "risk", "rolling std 60d of ret_1d"),
    ("downside_vol_20d", "risk", "sqrt(E[ret^2|ret<0] over 20d)"),
    ("beta_60d", "risk", "cov(ret,hs300)/var(hs300) over 60d"),
    ("idio_vol_60d", "risk", "rolling std of idiosyncratic return proxy"),
    ("turnover_mean_20d", "liquidity", "rolling mean turnover 20d"),
    ("amihud_20d", "liquidity", "rolling mean abs(ret)/amount 20d"),
    ("book_to_price", "valuation", "1/pb"),
    ("earnings_yield", "valuation", "1/pe_ttm"),
    ("sales_yield", "valuation", "1/ps_ttm"),
    ("cashflow_yield", "valuation", "1/pcf_ttm"),
    ("size_ln_mv", "size", "ln(total_mv)"),
    ("size_ln_mv_sq", "size", "ln(total_mv)^2"),
    ("ret_1d_cs_z", "cross_section", "cross-sectional zscore of daily return"),
    ("mom_20d_cs_z", "cross_section", "cross-sectional zscore of 20d momentum"),
    ("vol_20d_cs_z", "cross_section", "cross-sectional zscore of vol_20d"),
    ("size_cs_z", "cross_section", "cross-sectional zscore of size"),
    ("mom_20d_decile", "bucket", "cross-sectional decile by mom_20d"),
    ("value_decile", "bucket", "cross-sectional decile by book_to_price"),
]
dst.executemany("INSERT INTO factor_metadata VALUES (?, ?, ?)", metadata_rows)

counts = dst.execute("""
SELECT
  (SELECT COUNT(*) FROM trading_data_all) AS trading_rows,
  (SELECT COUNT(*) FROM fundamentals_snapshot) AS fundamentals_rows,
  (SELECT COUNT(*) FROM factor_zoo_daily) AS factor_rows,
  (SELECT COUNT(*) FROM factor_metadata) AS factor_count
""").fetchdf()

sample = dst.execute("""
SELECT * FROM factor_zoo_daily
ORDER BY trade_date DESC, stock_code
LIMIT 10
""").fetchdf()

print("\n[Build Summary]")
print(counts.to_string(index=False))
print("\n[Sample factors]")
print(sample.head(5).to_string(index=False))

dst.close()
src.close()
print("\n✅ Factor zoo has been built and saved to factor_zoo.duckdb")

4.3 BUILD FACTOR ZOO INTO NEW DUCKDB
Source DuckDB: D:\MFIN\7036\trading_data.duckdb
Target DuckDB: D:\MFIN\7036\factor_zoo.duckdb
Main trading table: trading_data_clean
Columns in main table: 24


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


[Build Summary]
 trading_rows  fundamentals_rows  factor_rows  factor_count
     15800649           15789323     15784240            32

[Sample factors]
trade_date stock_code stock_name    ret_1d     mom_5d   mom_10d   mom_20d   mom_60d  mom_120d  mom_252d      rev_5d   rev_20d  close_ma5_gap  close_ma10_gap  close_ma20_gap  close_ma60_gap  ma5_ma20_spread  ma20_ma60_spread  intraday_range  intraday_return  vol_20d  vol_60d  vol_120d  downside_vol_20d  beta_60d  idio_vol_60d  sharpe_252d  turnover_rate  turnover_mean_20d  turnover_std_20d  volume_mean_20d   amihud_20d  pe_ttm      pb  ps_ttm  pcf_ttm  dividend_yield  total_mv  float_mv  book_to_price  earnings_yield  sales_yield  cashflow_yield  size_ln_mv  size_ln_mv_sq  ret_1d_cs_z  mom_20d_cs_z  vol_20d_cs_z  size_cs_z  mom_20d_decile  value_decile
2025-12-05     000001       None  0.007000 336.905420  0.003655  0.017710 -0.012025  0.036361  0.155975 -336.905420 -0.017710       0.670777        0.998752        1.005582        0.973

In [ ]:
# Validate new factor_zoo.duckdb
from pathlib import Path
import duckdb

db_path = Path("factor_zoo.duckdb")
if not db_path.exists():
    raise FileNotFoundError("factor_zoo.duckdb not found in current folder")

con = duckdb.connect(str(db_path), read_only=True)
tables = con.execute("SHOW TABLES").fetchdf()
print("Tables in factor_zoo.duckdb:")
print(tables.to_string(index=False))

counts = con.execute("""
SELECT
  (SELECT COUNT(*) FROM trading_data_all) AS trading_rows,
  (SELECT COUNT(*) FROM fundamentals_snapshot) AS fundamentals_rows,
  (SELECT COUNT(*) FROM factor_zoo_daily) AS factor_rows,
  (SELECT COUNT(*) FROM factor_metadata) AS factor_count
""").fetchdf()
print("\nCore table counts:")
print(counts.to_string(index=False))

sample = con.execute("""
SELECT trade_date, stock_code, ret_1d, mom_20d, vol_20d, book_to_price, size_ln_mv
FROM factor_zoo_daily
WHERE ret_1d IS NOT NULL
ORDER BY trade_date DESC, stock_code
LIMIT 10
""").fetchdf()
print("\nSample factors:")
print(sample.to_string(index=False))
con.close()

Tables in factor_zoo.duckdb:
                 name
          factor_base
      factor_metadata
     factor_zoo_daily
fundamentals_snapshot
  raw_trading_data_en
 raw_trading_data_raw
     trading_data_all

Core table counts:
 trading_rows  fundamentals_rows  factor_rows  factor_count
     15800649           15789323     15784240            32

Sample factors:
trade_date stock_code    ret_1d   mom_20d  vol_20d  book_to_price  size_ln_mv
2025-12-05     000001  0.007000  0.017710 0.006945       0.680272         NaN
2025-12-05     000001  0.003481 -0.013687 0.008679       1.981768         NaN
2025-12-05     000002 -0.004024 -0.199029 0.020494       2.976190         NaN
2025-12-05     000004  0.015762 -0.066774 0.030977       0.024061         NaN
2025-12-05     000006  0.022290 -0.170912 0.023158       0.396197         NaN
2025-12-05     000007 -0.019704  0.028956 0.035659       0.052992         NaN
2025-12-05     000008  0.016667  0.030405 0.029815       0.353794         NaN
2025-12-05    